# DQL: String Functions

String functions clean, transform, and search text columns. Examples use employee names, jobs, departments, and project names.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Substring and Concatenation

Spark uses `substring` for part of a string and `concat` or `concat_ws` for joining strings.


In [ ]:
emp.select(
    "ename",
    F.substring("ename", 1, 5).alias("first_5_chars"),
    F.concat_ws(" - ", "ename", "job").alias("employee_label")
).show(10)

spark.sql("""
SELECT ename,
       SUBSTRING(ename, 1, 5) AS first_5_chars,
       CONCAT(ename, ' - ', job) AS employee_label
FROM emp
LIMIT 10
""").show(truncate=False)


## LOWER and UPPER

Use these functions to normalize text casing.


In [ ]:
emp.select("ename", F.lower("ename").alias("lower_name"), F.upper("job").alias("upper_job")).show(10)

spark.sql("""
SELECT ename, LOWER(ename) AS lower_name, UPPER(job) AS upper_job
FROM emp
LIMIT 10
""").show()


## TRIM, LTRIM, and RTRIM

Trim functions remove unwanted spaces. The sample data is already clean, so this creates a small demo DataFrame.


In [ ]:
messy = spark.createDataFrame([("  accounting  ",), ("  sales",), ("research  ",)], ["raw_text"])

messy.select(
    "raw_text",
    F.trim("raw_text").alias("trimmed"),
    F.ltrim("raw_text").alias("left_trimmed"),
    F.rtrim("raw_text").alias("right_trimmed")
).show(truncate=False)

messy.createOrReplaceTempView("messy")
spark.sql("""
SELECT raw_text, TRIM(raw_text) AS trimmed, LTRIM(raw_text) AS left_trimmed, RTRIM(raw_text) AS right_trimmed
FROM messy
""").show(truncate=False)


## CHARINDEX Equivalent

In Spark, use `instr` or `locate` to find the position of text inside a string.


In [ ]:
dept.select("dname", F.instr("dname", "a").alias("position_of_a")).show()

spark.sql("""
SELECT dname, INSTR(dname, 'a') AS position_of_a
FROM dept
""").show()


## LEFT and RIGHT

Spark SQL has `left` and `right`. In the DataFrame API, `substring` is portable and explicit.


In [ ]:
projects.select(
    "project_name",
    F.substring("project_name", 1, 3).alias("left_3"),
    F.expr("right(project_name, 3)").alias("right_3")
).show()

spark.sql("""
SELECT project_name, LEFT(project_name, 3) AS left_3, RIGHT(project_name, 3) AS right_3
FROM projects
""").show()


## REVERSE, REPLACE, and LENGTH

These functions are useful for formatting and data quality checks.


In [ ]:
emp.select(
    "ename",
    F.reverse("ename").alias("reversed_name"),
    F.regexp_replace("ename", "_", "-").alias("dash_name"),
    F.length("ename").alias("name_length")
).show(10)

spark.sql("""
SELECT ename,
       REVERSE(ename) AS reversed_name,
       REPLACE(ename, '_', '-') AS dash_name,
       LENGTH(ename) AS name_length
FROM emp
LIMIT 10
""").show()
